# Earned Wage Access (EWA) & CFPB/TILA Compliance - Interactive Walkthrough

## Overview & Regulatory Context

**Regulation**: CFPB Oversight / Truth in Lending Act (TILA)  
**Regulators**: CFPB, State Regulators, Department of Labor  
**Key Challenge**: Uncertain and evolving regulatory classification of EWA products

### Critical Compliance Challenge
Earned Wage Access (EWA) products exist in a regulatory gray area with evolving classification:
- **Uncertain Classification**: May be classified as credit, payment advance, or employer benefit
- **Evolving TILA Requirements**: Disclosure requirements change based on regulatory interpretation
- **APR Calculation Complexity**: Fee structures may require APR disclosure depending on classification
- **Consumer Protection Concerns**: CFPB focus on ability-to-repay and fee transparency
- **Temporal Tracking**: Regulatory guidance evolves over time, requiring historical compliance validation

### Learning Objectives
1. Implement EWA eligibility decisions with regulatory classification tracking
2. Calculate and disclose APR based on evolving TILA requirements
3. Document ability-to-repay assessments and payroll integration verification
4. Track regulatory guidance evolution over time for compliance consistency

## Setup & SDK Initialization

Let's start by importing the necessary modules and initializing the Briefcase AI SDK:

In [ ]:
import sys
import os
import uuid
import random
from datetime import datetime, timedelta
from typing import Dict, Any, List, Tuple

# Add shared module to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'shared'))

try:
    import backend
    from backend import briefcase_ai, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK and backend utilities")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")
    print("Please ensure the shared backend module is available")

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase_ai.init_with_config(2)  # Initialize with 2 worker threads
    print("[SUCCESS] Briefcase AI SDK initialized successfully")
    
    # Get configured backend for audit trail storage
    db_backend = backend.get_backend()
    print("[SUCCESS] SQLite backend configured for immutable audit storage")
    
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

## Regulatory Classification Tracking Engine

EWA products have uncertain and evolving regulatory classification. This function tracks guidance over time:

In [ ]:
def get_regulatory_classification_status(current_date: datetime) -> Dict[str, Any]:
    """
    Returns current regulatory classification status for EWA products.
    This changes over time as CFPB guidance evolves.
    """
    # Simulate evolving regulatory guidance over time
    if current_date < datetime(2023, 1, 1):
        return {
            "primary_classification": "uncertain",
            "potential_classifications": ["payroll_advance", "employer_benefit", "credit_product"],
            "tila_applicable": "uncertain",
            "cfpb_guidance_version": "2022-preliminary",
            "disclosure_requirements": ["basic_fee_disclosure"],
            "apr_calculation_required": False,
            "credit_reporting_required": False
        }
    elif current_date < datetime(2024, 1, 1):
        return {
            "primary_classification": "employer_benefit_with_credit_features",
            "potential_classifications": ["employer_benefit", "credit_product"],
            "tila_applicable": "conditional",
            "cfpb_guidance_version": "2023-interim",
            "disclosure_requirements": ["enhanced_fee_disclosure", "conditional_apr"],
            "apr_calculation_required": True,  # For disclosure purposes
            "credit_reporting_required": False
        }
    else:  # 2024 and later
        return {
            "primary_classification": "credit_product_with_exemptions",
            "potential_classifications": ["credit_product", "employer_benefit"],
            "tila_applicable": "yes_with_exemptions",
            "cfpb_guidance_version": "2024-final",
            "disclosure_requirements": ["full_tila_disclosure", "apr_calculation", "ability_to_repay"],
            "apr_calculation_required": True,
            "credit_reporting_required": True
        }

# Demonstrate regulatory evolution
current_date = datetime.utcnow()
current_status = get_regulatory_classification_status(current_date)

print("🏛 Current Regulatory Classification Status for EWA:")
print(f"   **Details:** Primary Classification: {current_status['primary_classification']}")
print(f"   **Reference:** CFPB Guidance Version: {current_status['cfpb_guidance_version']}")
print(f"   ⚖ TILA Applicable: {current_status['tila_applicable']}")
print(f"   **Results:** APR Calculation Required: {'[SUCCESS] YES' if current_status['apr_calculation_required'] else '[FAILED] NO'}")
print(f"   **Note:** Required Disclosures: {', '.join(current_status['disclosure_requirements'])}")

print("\n[WARNING] REGULATORY UNCERTAINTY:")
print("   EWA classification continues to evolve with CFPB guidance")
print("   Audit trails must preserve classification at decision time")

print("\n[SUCCESS] Regulatory Classification Tracking Engine defined")

## Earned Wage Calculation Engine

Calculate earned wages available for advance based on payroll integration:

In [ ]:
def calculate_earned_wages(employee_data: Dict[str, Any], current_date: datetime) -> Dict[str, Any]:
    """
    Calculates earned wages available for advance based on payroll data.
    This is the core calculation that determines advance eligibility.
    """
    # Parse pay period dates
    pay_period_start = datetime.fromisoformat(employee_data["pay_period_start"])
    next_payday = datetime.fromisoformat(employee_data["next_payday"])
    
    # Calculate days worked in current period
    days_worked = (current_date - pay_period_start).days
    total_pay_period_days = (next_payday - pay_period_start).days
    
    print(f"📅 Pay Period Analysis:")
    print(f"   Period Start: {pay_period_start.strftime('%Y-%m-%d')}")
    print(f"   Next Payday: {next_payday.strftime('%Y-%m-%d')}")
    print(f"   Days Worked: {days_worked} / {total_pay_period_days}")
    
    # Calculate earned wages based on employment type
    if employee_data.get("employment_type") == "hourly":
        hours_worked = employee_data.get("hours_worked_this_period", 0)
        hourly_rate = employee_data.get("hourly_rate", 0)
        gross_earned = hours_worked * hourly_rate
        print(f"   **Business:** Hourly Employee: {hours_worked} hours @ ${hourly_rate}/hour")
    else:  # Salaried
        annual_salary = employee_data.get("annual_salary", 0)
        daily_rate = annual_salary / 365
        gross_earned = daily_rate * days_worked
        print(f"   **Business:** Salaried Employee: ${annual_salary:,}/year (${daily_rate:.2f}/day)")
    
    print(f"   **Financial:** Gross Earned: ${gross_earned:.2f}")
    
    # Estimate taxes and deductions (simplified)
    tax_rate = employee_data.get("estimated_tax_rate", 0.22)  # Federal + state + FICA
    other_deductions = employee_data.get("other_deductions_per_period", 0)
    
    estimated_taxes = gross_earned * tax_rate
    net_earned = gross_earned - estimated_taxes - (other_deductions * days_worked / total_pay_period_days)
    
    print(f"   **Results:** Tax Withholding ({tax_rate*100:.1f}%): ${estimated_taxes:.2f}")
    print(f"   💵 Net Earned: ${net_earned:.2f}")
    
    # Apply EWA platform constraints
    max_advance_percentage = 0.5  # Maximum 50% of net earned
    max_advance_dollar = min(500, net_earned * max_advance_percentage)  # $500 platform limit
    
    print(f"   **Objective:** Max Advance Available: ${max_advance_dollar:.2f} (50% of net earned, $500 limit)")
    
    return {
        "gross_earned_wages": round(gross_earned, 2),
        "estimated_net_earned": round(net_earned, 2),
        "max_advance_available": round(max_advance_dollar, 2),
        "days_worked_current_period": days_worked,
        "total_pay_period_days": total_pay_period_days,
        "earnings_calculation_method": "hourly" if employee_data.get("employment_type") == "hourly" else "salary_pro_rata",
        "tax_withholding_estimated": round(estimated_taxes, 2)
    }

print("[SUCCESS] Earned Wage Calculation Engine defined")
print("   **Business:** Supports both hourly and salaried employees")
print("   🧮 Includes tax withholding and deduction estimates")
print("   **Objective:** Applies platform limits and percentage constraints")

## Fee Structure & APR Calculation Engine

Calculate fees and APR based on current TILA requirements:

In [ ]:
def calculate_ewa_fees_and_apr(advance_amount: float, repayment_date: datetime, current_date: datetime) -> Dict[str, Any]:
    """
    Calculates EWA fees and APR based on current regulatory requirements.
    APR calculation may be required for TILA disclosure depending on classification.
    """
    days_to_repayment = (repayment_date - current_date).days
    
    print(f"**Financial:** Fee Calculation for ${advance_amount:.2f}:")
    print(f"   📅 Days to Repayment: {days_to_repayment}")
    
    # Fee structure based on industry standards
    if advance_amount <= 100:
        base_fee = 2.99
    elif advance_amount <= 250:
        base_fee = 3.99
    else:
        base_fee = 4.99
    
    # Optional expedited fee (simulated)
    expedited_fee = 1.99 if random.choice([True, False]) else 0
    
    total_fee = base_fee + expedited_fee
    
    print(f"   **Results:** Base Fee: ${base_fee:.2f}")
    if expedited_fee > 0:
        print(f"   **Launch:** Expedited Fee: ${expedited_fee:.2f}")
    print(f"   **Cost:** Total Fee: ${total_fee:.2f}")
    
    # APR calculation for disclosure (annualized)
    if days_to_repayment > 0 and advance_amount > 0:
        apr = ((total_fee / advance_amount) * (365 / days_to_repayment))
        fee_percentage = (total_fee / advance_amount) * 100
        
        print(f"   **Metrics:** Calculated APR: {apr:.2%} (for disclosure purposes)")
        print(f"   **Results:** Fee as Percentage: {fee_percentage:.2f}%")
    else:
        apr = 0
        fee_percentage = 0
    
    return {
        "base_fee": base_fee,
        "expedited_fee": expedited_fee,
        "total_fee": total_fee,
        "advance_amount": advance_amount,
        "repayment_amount": advance_amount,  # Principal only (fees separate)
        "days_to_repayment": days_to_repayment,
        "calculated_apr": round(apr, 4),
        "fee_as_percentage": round(fee_percentage, 2)
    }

print("[SUCCESS] Fee Structure & APR Calculation Engine defined")
print("   **Financial:** Implements tiered fee structure based on advance amount")
print("   **Metrics:** Calculates APR for TILA disclosure requirements")
print("   **Launch:** Supports optional expedited processing fees")

## Ability-to-Repay Assessment Engine

Assess employee's ability to repay the advance (required by some regulatory frameworks):

In [ ]:
def assess_ability_to_repay(employee_data: Dict[str, Any], advance_amount: float) -> Dict[str, Any]:
    """
    Assesses employee's ability to repay the advance based on income and obligations.
    This may be required depending on regulatory classification.
    """
    print(f"**Analysis:** Ability-to-Repay Assessment for ${advance_amount:.2f}:")
    
    # Calculate monthly income
    if employee_data.get("employment_type") == "hourly":
        monthly_income = employee_data.get("hourly_rate", 0) * 40 * 4.33  # Assume 40 hrs/week
        print(f"   **Business:** Hourly Income: ${employee_data.get('hourly_rate', 0)}/hour → ${monthly_income:.2f}/month")
    else:
        monthly_income = employee_data.get("annual_salary", 0) / 12
        print(f"   **Business:** Salaried Income: ${employee_data.get('annual_salary', 0):,}/year → ${monthly_income:.2f}/month")
    
    # Estimate monthly expenses (if not provided)
    estimated_monthly_expenses = employee_data.get("monthly_expenses", monthly_income * 0.8)
    print(f"   **Payment:** Monthly Expenses: ${estimated_monthly_expenses:.2f}")
    
    # Calculate discretionary income
    discretionary_income = monthly_income - estimated_monthly_expenses
    print(f"   **Financial:** Discretionary Income: ${discretionary_income:.2f}")
    
    # Check previous EWA usage
    previous_advances = employee_data.get("previous_advances_count", 0)
    previous_advance_amount = employee_data.get("previous_advance_outstanding", 0)
    
    print(f"   **Results:** Previous Advances: {previous_advances} (Outstanding: ${previous_advance_amount:.2f})")
    
    # Calculate repayment capacity
    available_for_repayment = discretionary_income + previous_advance_amount  # Previous will be repaid
    repayment_ratio = advance_amount / available_for_repayment if available_for_repayment > 0 else float('inf')
    
    print(f"   **Objective:** Available for Repayment: ${available_for_repayment:.2f}")
    print(f"   **Metrics:** Repayment Ratio: {repayment_ratio:.3f}")
    
    # Determine ability to repay based on criteria
    criteria = [
        ("repayment_ratio_acceptable", repayment_ratio <= 0.5, "Advance ≤ 50% of available income"),
        ("frequency_acceptable", previous_advances <= 3, "≤ 3 previous advances"),
        ("amount_within_limits", advance_amount <= 500, "≤ $500 advance limit"),
        ("minimum_income_met", monthly_income >= 2000, "≥ $2,000 monthly income")
    ]
    
    print(f"\n   [SUCCESS] Ability-to-Repay Criteria:")
    can_repay = True
    for criterion_name, criterion_met, description in criteria:
        status = "[SUCCESS] PASS" if criterion_met else "[FAILED] FAIL"
        print(f"      {description}: {status}")
        if not criterion_met:
            can_repay = False
    
    print(f"   **Achievement:** Overall Assessment: {'[SUCCESS] CAN REPAY' if can_repay else '[FAILED] CANNOT REPAY'}")
    
    return {
        "monthly_income": round(monthly_income, 2),
        "estimated_monthly_expenses": round(estimated_monthly_expenses, 2),
        "discretionary_income": round(discretionary_income, 2),
        "available_for_repayment": round(available_for_repayment, 2),
        "repayment_ratio": round(repayment_ratio, 3),
        "can_repay_determination": can_repay,
        "previous_advances_count": previous_advances,
        "income_verification_status": employee_data.get("income_verified", True)
    }

print("[SUCCESS] Ability-to-Repay Assessment Engine defined")
print("   **Financial:** Calculates discretionary income and repayment capacity")
print("   **Results:** Applies multiple criteria for comprehensive assessment")
print("   **Metrics:** Considers previous advance history and frequency")

## EWA Eligibility Decision Engine

The core AI system that makes EWA eligibility decisions with full regulatory compliance:

In [ ]:
def simulate_ewa_eligibility_decision(employee_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates an AI decision for earned wage access eligibility with full compliance analysis.
    In production, this would be replaced with actual ML model inference.
    """
    current_date = datetime.utcnow()
    request_amount = employee_data.get("advance_request_amount", 0)
    
    print(f"\n[AUTOMATED] EWA Eligibility Analysis for Employee {employee_data['employee_id'][:8]}...")
    print(f"   👤 Employee: {employee_data['employee_name']}")
    print(f"   **Institution:** Employer: {employee_data['employer']}")
    print(f"   **Financial:** Requested Amount: ${request_amount:.2f}")
    
    # Get current regulatory classification (CRITICAL for compliance)
    print(f"\n🏛 Determining Regulatory Classification...")
    regulatory_status = get_regulatory_classification_status(current_date)
    print(f"   **Details:** Current Classification: {regulatory_status['primary_classification']}")
    print(f"   **Reference:** CFPB Guidance: {regulatory_status['cfpb_guidance_version']}")
    print(f"   ⚖ TILA Applicable: {regulatory_status['tila_applicable']}")
    
    # Calculate earned wages
    print(f"\n**Business:** Calculating Earned Wages...")
    wage_calculation = calculate_earned_wages(employee_data, current_date)
    
    # Assess basic eligibility criteria
    print(f"\n[SUCCESS] Evaluating Eligibility Criteria...")
    eligibility_factors = []
    
    # Employment verification
    if employee_data.get("employer_verified", False):
        eligibility_factors.append("employer_verified")
        print(f"   [SUCCESS] Employer Verified")
    else:
        eligibility_factors.append("employer_not_verified")
        print(f"   [FAILED] Employer Not Verified")
    
    # Wage availability
    if wage_calculation["max_advance_available"] >= request_amount:
        eligibility_factors.append("sufficient_earned_wages")
        print(f"   [SUCCESS] Sufficient Earned Wages (${wage_calculation['max_advance_available']:.2f} available)")
    else:
        eligibility_factors.append("insufficient_earned_wages")
        print(f"   [FAILED] Insufficient Earned Wages (${wage_calculation['max_advance_available']:.2f} < ${request_amount:.2f})")
    
    # Account standing
    account_standing = employee_data.get("account_standing", "good")
    eligibility_factors.append(f"account_standing_{account_standing}")
    print(f"   {'[SUCCESS]' if account_standing in ['good', 'excellent'] else '[FAILED]'} Account Standing: {account_standing.title()}")
    
    # Previous advance history
    previous_count = employee_data.get("previous_advances_count", 0)
    if previous_count <= 2:
        eligibility_factors.append("acceptable_advance_history")
        print(f"   [SUCCESS] Acceptable Advance History ({previous_count} previous)")
    else:
        eligibility_factors.append("excessive_advance_frequency")
        print(f"   [FAILED] Excessive Advance Frequency ({previous_count} previous)")
    
    # Determine eligibility
    disqualifying_factors = [
        "employer_not_verified",
        "insufficient_earned_wages", 
        "account_standing_poor",
        "excessive_advance_frequency"
    ]
    
    is_eligible = not any(factor in eligibility_factors for factor in disqualifying_factors)
    
    print(f"\n**Objective:** Eligibility Determination: {'[SUCCESS] ELIGIBLE' if is_eligible else '[FAILED] NOT ELIGIBLE'}")
    
    if is_eligible:
        # Determine advance amount (may be less than requested)
        max_available = wage_calculation["max_advance_available"]
        approved_amount = min(request_amount, max_available, 500)  # $500 platform limit
        
        print(f"   **Financial:** Approved Amount: ${approved_amount:.2f}")
        
        # Calculate fees and APR
        next_payday = datetime.fromisoformat(employee_data["next_payday"])
        fee_calculation = calculate_ewa_fees_and_apr(approved_amount, next_payday, current_date)
        
        # Assess ability to repay (required by some regulatory frameworks)
        repay_assessment = assess_ability_to_repay(employee_data, approved_amount)
        
        eligibility_status = "approved"
        decline_reasons = []
        
        # Final check: ability to repay
        if not repay_assessment["can_repay_determination"]:
            eligibility_status = "conditional"  # May require additional verification
            print(f"   [WARNING] Status Changed to CONDITIONAL due to repayment concerns")
            
    else:
        approved_amount = 0
        fee_calculation = calculate_ewa_fees_and_apr(0, current_date, current_date)
        repay_assessment = assess_ability_to_repay(employee_data, 0)
        eligibility_status = "declined"
        decline_reasons = [factor for factor in eligibility_factors if factor in disqualifying_factors]
        
        print(f"   [FAILED] Decline Reasons: {', '.join(decline_reasons)}")
    
    # Determine required disclosures based on regulatory status
    required_disclosures = []
    if regulatory_status["tila_applicable"] in ["yes_with_exemptions", "conditional"]:
        required_disclosures.extend(["tila_disclosure", "apr_disclosure"])
    if "full_tila_disclosure" in regulatory_status["disclosure_requirements"]:
        required_disclosures.extend(["full_tila_disclosure", "ability_to_repay_disclosure"])
    
    required_disclosures.extend(["fee_structure_disclosure", "repayment_terms_disclosure"])
    
    print(f"\n**Details:** Required Disclosures: {', '.join(required_disclosures)}")
    
    return {
        "eligibility_status": eligibility_status,
        "approved_advance_amount": approved_amount,
        "requested_amount": request_amount,
        "max_available_amount": wage_calculation["max_advance_available"],
        "advance_fee": fee_calculation["total_fee"],
        "repayment_amount": approved_amount,  # EWA typically has no interest
        "repayment_date": employee_data["next_payday"],
        "calculated_apr": fee_calculation["calculated_apr"],
        "fee_as_percentage": fee_calculation["fee_as_percentage"],
        "wage_calculation": wage_calculation,
        "fee_breakdown": {
            "base_fee": fee_calculation["base_fee"],
            "expedited_fee": fee_calculation["expedited_fee"],
            "total_fee": fee_calculation["total_fee"]
        },
        "eligibility_factors": eligibility_factors,
        "decline_reasons": decline_reasons,
        "ability_to_repay": repay_assessment,
        "regulatory_classification": regulatory_status,
        "required_disclosures": required_disclosures,
        "tila_disclosures_required": regulatory_status["tila_applicable"] != "uncertain",
        "employer_integration_verified": employee_data.get("employer_verified", False),
        "payroll_deduction_authorized": employee_data.get("payroll_deduction_consent", False),
        "model_version": "ewa-eligibility-v3.1.2",
        "decision_trace_id": str(uuid.uuid4()),
        "decision_timestamp": current_date.isoformat()
    }

print("[SUCCESS] EWA Eligibility Decision Engine defined")
print("   **Objective:** Balances employee needs with regulatory compliance")
print("   **Details:** Tracks regulatory classification evolution over time")
print("   **Financial:** Implements comprehensive eligibility and repayment assessment")
print("   **Results:** Provides complete TILA disclosure when required")

## EWA Scenario Processing

Let's process different EWA scenarios to demonstrate regulatory compliance:

In [ ]:
# Define realistic EWA scenarios across different employee types
ewa_scenarios = [
    {
        "scenario_name": "Hourly Food Service Worker - High Frequency User",
        "employee_data": {
            "employee_id": str(uuid.uuid4()),
            "employee_name": "Maria Santos",
            "employer": "Restaurant Chain Inc",
            "employment_type": "hourly",
            "hourly_rate": 16.50,
            "hours_worked_this_period": 35,
            "pay_period_start": "2024-02-12",
            "next_payday": "2024-02-26",
            "advance_request_amount": 180.00,
            "previous_advances_count": 3,  # High frequency user
            "previous_advance_outstanding": 0,
            "account_standing": "good",
            "employer_verified": True,
            "payroll_deduction_consent": True,
            "estimated_tax_rate": 0.18,
            "monthly_expenses": 2400,
            "income_verified": True
        }
    },
    {
        "scenario_name": "Salaried Office Worker - First Time User",
        "employee_data": {
            "employee_id": str(uuid.uuid4()),
            "employee_name": "David Chen",
            "employer": "Tech Solutions LLC",
            "employment_type": "salaried",
            "annual_salary": 55000,
            "pay_period_start": "2024-02-01",
            "next_payday": "2024-03-01",
            "advance_request_amount": 300.00,
            "previous_advances_count": 0,  # First time user
            "previous_advance_outstanding": 0,
            "account_standing": "excellent",
            "employer_verified": True,
            "payroll_deduction_consent": True,
            "estimated_tax_rate": 0.24,
            "monthly_expenses": 3200,
            "income_verified": True
        }
    }
]

print("**Details:** EWA Employee Scenarios:")
for i, scenario in enumerate(ewa_scenarios, 1):
    employee = scenario["employee_data"]
    print(f"\n{i}. {scenario['scenario_name']}")
    print(f"   👤 {employee['employee_name']} at {employee['employer']}")
    print(f"   **Business:** Type: {employee['employment_type'].title()}")
    if employee['employment_type'] == 'hourly':
        print(f"   **Financial:** Rate: ${employee['hourly_rate']}/hour ({employee['hours_worked_this_period']} hours worked)")
    else:
        print(f"   **Financial:** Salary: ${employee['annual_salary']:,}/year")
    print(f"   📅 Request: ${employee['advance_request_amount']:.2f}")
    print(f"   **Results:** Previous Advances: {employee['previous_advances_count']}")

print("\n**Objective:** These scenarios test different aspects of EWA compliance")

decision_ids = []  # Track for audit demonstration

### Process Scenario 1: High Frequency Hourly Worker

In [ ]:
print("=" * 60)
print("🍔 PROCESSING: Hourly Food Service Worker - High Frequency User")
print("=" * 60)

employee_data_1 = ewa_scenarios[0]["employee_data"]

# Run AI EWA eligibility decision
ewa_decision_1 = simulate_ewa_eligibility_decision(employee_data_1)

print(f"\n**Results:** FINAL EWA DECISION:")
print(f"   **Objective:** Eligibility Status: {ewa_decision_1['eligibility_status'].upper()}")
print(f"   **Financial:** Approved Amount: ${ewa_decision_1['approved_advance_amount']:.2f}")
print(f"   **Metrics:** Max Available: ${ewa_decision_1['max_available_amount']:.2f}")

if ewa_decision_1["approved_advance_amount"] > 0:
    print(f"   **Cost:** Total Fee: ${ewa_decision_1['advance_fee']:.2f}")
    print(f"   **Results:** APR (for disclosure): {ewa_decision_1['calculated_apr']:.1%}")
    print(f"   **Metrics:** Fee Percentage: {ewa_decision_1['fee_as_percentage']:.1f}%")
    print(f"   📅 Repayment Date: {ewa_decision_1['repayment_date']}")

# Display regulatory classification
reg_class = ewa_decision_1["regulatory_classification"]
print(f"\n🏛 Regulatory Classification:")
print(f"   **Details:** Classification: {reg_class['primary_classification']}")
print(f"   **Reference:** CFPB Guidance: {reg_class['cfpb_guidance_version']}")
print(f"   ⚖ TILA Applicable: {reg_class['tila_applicable']}")
print(f"   **Results:** APR Required: {'[SUCCESS] YES' if reg_class['apr_calculation_required'] else '[FAILED] NO'}")

# Display eligibility factors or decline reasons
if ewa_decision_1["decline_reasons"]:
    print(f"\n[FAILED] Decline Reasons: {', '.join(ewa_decision_1['decline_reasons'])}")

if ewa_decision_1["required_disclosures"]:
    print(f"\n**Details:** Required Disclosures: {', '.join(ewa_decision_1['required_disclosures'])}")

# Display ability to repay assessment
repay_assessment = ewa_decision_1["ability_to_repay"]
print(f"\n**Financial:** Ability to Repay Assessment:")
print(f"   [SUCCESS] Can Repay: {'YES' if repay_assessment['can_repay_determination'] else 'NO'}")
print(f"   **Results:** Repayment Ratio: {repay_assessment['repayment_ratio']:.3f}")
print(f"   💵 Monthly Income: ${repay_assessment['monthly_income']:.2f}")
print(f"   **Payment:** Monthly Expenses: ${repay_assessment['estimated_monthly_expenses']:.2f}")
print(f"   **Financial:** Discretionary Income: ${repay_assessment['discretionary_income']:.2f}")

### Create Audit Trail for High Frequency Worker

In [ ]:
# Create comprehensive regulatory metadata for EWA
regulatory_metadata_1 = {
    "regulation": "CFPB/TILA",
    "regulatory_classification": reg_class["primary_classification"],
    "cfpb_guidance_version": reg_class["cfpb_guidance_version"],
    "tila_applicable": reg_class["tila_applicable"],
    "tila_disclosures_provided": ewa_decision_1["tila_disclosures_required"],
    "apr_calculated": ewa_decision_1["calculated_apr"] if ewa_decision_1["approved_advance_amount"] > 0 else None,
    "fee_structure_disclosed": True,
    "ability_to_repay_assessed": True,
    "payroll_integration_verified": ewa_decision_1["employer_integration_verified"],
    "consumer_protection_compliant": True,
    "advance_approved": ewa_decision_1["eligibility_status"] == "approved",
    "employer_consent_verified": employee_data_1.get("employer_verified", False),
    "payroll_deduction_authorized": employee_data_1.get("payroll_deduction_consent", False),
    "wage_calculation_method": ewa_decision_1["wage_calculation"]["earnings_calculation_method"],
    "regulatory_uncertainty_acknowledged": reg_class["primary_classification"] in ["uncertain", "employer_benefit_with_credit_features"],
    "examiner_ready": True,
    "decision_timestamp": datetime.utcnow().isoformat()
}

print("💾 Creating EWA Compliance Audit Trail...")
print("**Details:** Regulatory Metadata:")
for key, value in regulatory_metadata_1.items():
    if isinstance(value, bool):
        status = "[SUCCESS] YES" if value else "[FAILED] NO"
        print(f"   {key.replace('_', ' ').title()}: {status}")
    elif value is not None:
        print(f"   {key.replace('_', ' ').title()}: {value}")

# Create DecisionSnapshot with detailed type information
try:
    decision_snapshot_1 = backend.create_decision_snapshot(
        function_name="ewa_non_traditional_credit",
        inputs=employee_data_1,
        outputs=ewa_decision_1,
        metadata=regulatory_metadata_1,
        input_types={
            "hourly_rate": "float",
            "hours_worked_this_period": "int",
            "advance_request_amount": "float",
            "previous_advances_count": "int",
            "previous_advance_outstanding": "float",
            "estimated_tax_rate": "float",
            "monthly_expenses": "float",
            "employer_verified": "bool",
            "payroll_deduction_consent": "bool",
            "income_verified": "bool"
        },
        output_types={
            "approved_advance_amount": "float",
            "requested_amount": "float",
            "max_available_amount": "float",
            "advance_fee": "float",
            "calculated_apr": "float",
            "fee_as_percentage": "float",
            "tila_disclosures_required": "bool"
        }
    )
    
    print(f"\n[SUCCESS] Decision snapshot created successfully")
    print(f"   **Results:** Regulatory classification preserved")
    print(f"   **Financial:** Wage calculation methodology documented")
    
except Exception as e:
    print(f"[FAILED] Error creating decision snapshot: {e}")

# Store decision in audit trail
try:
    stored_decision_id_1 = db_backend.save_decision(decision_snapshot_1)
    decision_ids.append(stored_decision_id_1)
    
    print(f"[SUCCESS] Decision stored in immutable audit trail: {stored_decision_id_1[:12]}...")
    print("[PROTECTED] Complete EWA compliance documentation preserved")
    print("⚖ CFPB examination ready with regulatory classification tracking")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

### Process Scenario 2: First-Time Salaried User

In [ ]:
print("\n" + "=" * 60)
print("[SYSTEM] PROCESSING: Salaried Office Worker - First Time User")
print("=" * 60)

employee_data_2 = ewa_scenarios[1]["employee_data"]

# Run AI EWA eligibility decision
ewa_decision_2 = simulate_ewa_eligibility_decision(employee_data_2)

print(f"\n**Results:** FINAL EWA DECISION:")
print(f"   **Objective:** Eligibility Status: {ewa_decision_2['eligibility_status'].upper()}")
print(f"   **Financial:** Approved Amount: ${ewa_decision_2['approved_advance_amount']:.2f}")

if ewa_decision_2["approved_advance_amount"] > 0:
    print(f"   **Cost:** Total Fee: ${ewa_decision_2['advance_fee']:.2f}")
    print(f"   **Results:** APR (for disclosure): {ewa_decision_2['calculated_apr']:.1%}")
    print(f"   **Metrics:** Fee Percentage: {ewa_decision_2['fee_as_percentage']:.1f}%")

# Display regulatory classification
reg_class_2 = ewa_decision_2["regulatory_classification"]
print(f"\n🏛 Regulatory Classification:")
print(f"   **Details:** Classification: {reg_class_2['primary_classification']}")
print(f"   **Reference:** CFPB Guidance: {reg_class_2['cfpb_guidance_version']}")

if ewa_decision_2["required_disclosures"]:
    print(f"\n**Details:** Required Disclosures: {', '.join(ewa_decision_2['required_disclosures'])}")

# Create and store audit trail (simplified for brevity)
regulatory_metadata_2 = {
    "regulation": "CFPB/TILA",
    "regulatory_classification": reg_class_2["primary_classification"],
    "cfpb_guidance_version": reg_class_2["cfpb_guidance_version"],
    "tila_applicable": reg_class_2["tila_applicable"],
    "tila_disclosures_provided": ewa_decision_2["tila_disclosures_required"],
    "apr_calculated": ewa_decision_2["calculated_apr"] if ewa_decision_2["approved_advance_amount"] > 0 else None,
    "fee_structure_disclosed": True,
    "ability_to_repay_assessed": True,
    "payroll_integration_verified": ewa_decision_2["employer_integration_verified"],
    "consumer_protection_compliant": True,
    "advance_approved": ewa_decision_2["eligibility_status"] == "approved",
    "employer_consent_verified": employee_data_2.get("employer_verified", False),
    "payroll_deduction_authorized": employee_data_2.get("payroll_deduction_consent", False),
    "wage_calculation_method": ewa_decision_2["wage_calculation"]["earnings_calculation_method"],
    "regulatory_uncertainty_acknowledged": reg_class_2["primary_classification"] in ["uncertain", "employer_benefit_with_credit_features"],
    "examiner_ready": True,
    "decision_timestamp": datetime.utcnow().isoformat()
}

decision_snapshot_2 = backend.create_decision_snapshot(
    function_name="ewa_non_traditional_credit",
    inputs=employee_data_2,
    outputs=ewa_decision_2,
    metadata=regulatory_metadata_2
)

stored_decision_id_2 = db_backend.save_decision(decision_snapshot_2)
decision_ids.append(stored_decision_id_2)

print(f"\n[SUCCESS] Decision stored in audit trail: {stored_decision_id_2[:12]}...")
print("[PROTECTED] Salaried employee wage calculation documented")
print("**Results:** First-time user eligibility validation preserved")

## CFPB Examiner Query Simulation

Demonstrate how to respond to regulatory examination queries for EWA compliance:

In [ ]:
print("\n" + "=" * 60)
print("👨‍**Business:** CFPB EXAMINER SIMULATION - EWA COMPLIANCE")
print("=" * 60)

cfpb_ewa_queries = [
    "Demonstrate TILA disclosure compliance and APR calculation methodology for EWA products",
    "Show evidence of regulatory classification tracking and guidance evolution over time",
    "Provide audit trail for ability-to-repay assessments and payroll integration verification",
    "Document fee structure compliance and consumer protection measures for wage advances"
]

for i, query in enumerate(cfpb_ewa_queries):
    if i < len(decision_ids):
        print(f"\n**Details:** EXAMINER QUERY {i+1}:")
        print(f"   {query}")
        print()
        
        response = backend.format_examiner_response(decision_ids[i], query, db_backend)
        print("🏛 REGULATORY RESPONSE:")
        print(response)
    else:
        print(f"\n**Details:** EXAMINER QUERY {i+1}: {query}")
        print("   (Additional decisions would be available in full implementation)")

print("\n**Objective:** EXAMINATION BENEFITS:")
print("   [SUCCESS] Immediate response capability for complex CFPB EWA queries")
print("   [SUCCESS] Complete TILA disclosure and APR calculation documentation")
print("   [SUCCESS] Regulatory classification evolution tracking over time")
print("   [SUCCESS] Ability-to-repay assessment and payroll verification evidence")
print("   [SUCCESS] Fee structure transparency and consumer protection validation")

## Regulatory Classification Evolution Tracking

Demonstrate how EWA regulatory classification tracking works over time:

In [ ]:
print("\n" + "=" * 60)
print("**Metrics:** REGULATORY CLASSIFICATION EVOLUTION TRACKING")
print("=" * 60)

print("🏛 EWA Regulatory Classification Evolution:")
print()

# Show how classification has evolved over time
classification_timeline = [
    (datetime(2022, 6, 1), "Pre-2023: Uncertain classification"),
    (datetime(2023, 6, 1), "2023: Employer benefit with credit features"),
    (datetime(2024, 6, 1), "2024+: Credit product with exemptions")
]

for date, description in classification_timeline:
    status = get_regulatory_classification_status(date)
    print(f"📅 {description}:")
    print(f"   **Details:** Classification: {status['primary_classification']}")
    print(f"   **Reference:** CFPB Guidance: {status['cfpb_guidance_version']}")
    print(f"   ⚖ TILA Applicable: {status['tila_applicable']}")
    print(f"   **Results:** APR Required: {'[SUCCESS] YES' if status['apr_calculation_required'] else '[FAILED] NO'}")
    print()

if decision_ids:
    print("**Details:** Current Decisions - Regulatory Classification Tracking:")
    
    for i, decision_id in enumerate(decision_ids):
        decision = db_backend.load_decision(decision_id)
        if decision:
            scenario_name = ewa_scenarios[i]["scenario_name"] if i < len(ewa_scenarios) else f"Decision {i+1}"
            
            classification = decision.tags.get("regulatory_classification", "Unknown")
            guidance_version = decision.tags.get("cfpb_guidance_version", "Unknown")
            tila_status = decision.tags.get("tila_applicable", "Unknown")
            apr_calculated = decision.tags.get("apr_calculated", "N/A")
            
            print(f"\n📄 {scenario_name} ({decision_id[:8]}...):")
            print(f"   **Details:** Classification: {classification}")
            print(f"   **Reference:** CFPB Guidance: {guidance_version}")
            print(f"   ⚖ TILA Status: {tila_status}")
            print(f"   **Results:** APR Calculated: {apr_calculated if apr_calculated != 'N/A' else 'N/A'}")
    
    print(f"\n**Objective:** TEMPORAL CONSISTENCY BENEFITS:")
    print(f"   [SUCCESS] Classification preserved at decision time")
    print(f"   [SUCCESS] Guidance version evolution tracked")
    print(f"   [SUCCESS] TILA applicability documented historically")
    print(f"   [SUCCESS] APR calculation requirements preserved")
    
    print(f"\n**Insight:** REGULATORY SCENARIO:")
    print(f"   If CFPB issues new EWA guidance, Briefcase AI can:")
    print(f"   • Show which decisions used previous vs. current guidance")
    print(f"   • Prove compliance decisions were correct at time of action")
    print(f"   • Validate TILA disclosure requirements evolution")
    print(f"   • Support regulatory examination queries about classification changes")

else:
    print("\n[WARNING] No decisions available for regulatory classification tracking demonstration")

## Regulatory Compliance Validation

Validate compliance across all EWA decisions:

In [ ]:
print("\n" + "=" * 60)
print("⚖ REGULATORY COMPLIANCE VALIDATION")
print("=" * 60)

# Define required compliance fields for EWA
required_fields = [
    "regulation",
    "regulatory_classification",
    "tila_applicable",
    "fee_structure_disclosed",
    "ability_to_repay_assessed",
    "payroll_integration_verified",
    "consumer_protection_compliant",
    "employer_consent_verified",
    "wage_calculation_method"
]

print("**Details:** Required CFPB/TILA EWA Compliance Fields:")
for field in required_fields:
    print(f"   • {field.replace('_', ' ').title()}")

if decision_ids:
    compliant_count = 0
    total_decisions = len(decision_ids)
    
    print(f"\n**Results:** COMPLIANCE ANALYSIS:")
    
    for i, decision_id in enumerate(decision_ids):
        decision = db_backend.load_decision(decision_id)
        if decision:
            validation = backend.validate_regulatory_completeness(decision, required_fields)
            
            scenario_name = ewa_scenarios[i]["scenario_name"] if i < len(ewa_scenarios) else f"Decision {i+1}"
            
            print(f"\n📄 {scenario_name}:")
            print(f"   Decision ID: {decision_id[:12]}...")
            print(f"   Compliance Status: {'[SUCCESS] COMPLIANT' if validation['is_compliant'] else '[FAILED] NON-COMPLIANT'}")
            print(f"   Completeness Score: {validation['completeness_score']:.1%}")
            
            # Show specific compliance aspects
            classification = decision.tags.get('regulatory_classification', 'N/A')
            tila_status = decision.tags.get('tila_applicable', 'N/A')
            payroll_verified = decision.tags.get('payroll_integration_verified', False)
            
            print(f"   **Details:** Classification Tracked: {classification}")
            print(f"   ⚖ TILA Status: {tila_status}")
            print(f"   **Business:** Payroll Integration: {'[SUCCESS]' if payroll_verified else '[FAILED]'}")
            
            if validation['missing_fields']:
                print(f"   [FAILED] Missing Fields: {', '.join(validation['missing_fields'])}")
            
            if validation["is_compliant"]:
                compliant_count += 1
    
    overall_compliance_rate = (compliant_count / total_decisions) * 100
    
    print(f"\n**Achievement:** OVERALL COMPLIANCE SUMMARY:")
    print(f"   Compliant Decisions: {compliant_count}/{total_decisions}")
    print(f"   Overall Compliance Rate: {overall_compliance_rate:.1f}%")
    print(f"   CFPB Examination Readiness: {'[SUCCESS] READY' if overall_compliance_rate >= 95 else '[WARNING] NEEDS IMPROVEMENT'}")
    print(f"   TILA Disclosure Compliance: {'[SUCCESS] COMPLIANT' if overall_compliance_rate >= 90 else '[WARNING] REVIEW REQUIRED'}")
    print(f"   Consumer Protection Status: {'[SUCCESS] STRONG' if overall_compliance_rate == 100 else '[WARNING] MONITOR'}")

    print("\n[SUCCESS] All EWA decisions documented with regulatory classification tracking")
else:
    print("\n[WARNING] No decisions available for compliance validation")

## Value Summary for EWA Providers

Summarize the key benefits of using Briefcase AI for EWA compliance:

In [ ]:
print("\n" + "=" * 60)
print("**Premium:** BRIEFCASE AI VALUE FOR EARNED WAGE ACCESS PROVIDERS")
print("=" * 60)

benefits = [
    ("Regulatory Classification Tracking", "Preserves regulatory status at decision time as CFPB guidance evolves"),
    ("TILA Disclosure Compliance", "Automated APR calculation and disclosure based on current classification"),
    ("Ability-to-Repay Assessment", "Comprehensive financial capacity evaluation for consumer protection"),
    ("Payroll Integration Verification", "Documentation of employer consent and wage calculation accuracy"),
    ("Fee Structure Transparency", "Complete disclosure of all fees with percentage and APR calculations"),
    ("CFPB Examination Readiness", "Immediate response capability for regulatory queries and guidance changes"),
    ("Consumer Protection Compliance", "Automated safeguards against excessive advance frequency and amounts"),
    ("Temporal Regulatory Consistency", "Historical validation of decisions against applicable guidance at time of decision")
]

for benefit, description in benefits:
    print(f"\n[SUCCESS] {benefit}:")
    print(f"   → {description}")

# Summary statistics
if decision_ids:
    total_processed = len(decision_ids)
    
    # Calculate approval rate
    approved_count = 0
    for decision_id in decision_ids:
        decision = db_backend.load_decision(decision_id)
        if decision:
            advance_approved = decision.tags.get('advance_approved', False)
            if advance_approved:
                approved_count += 1
    
    approval_rate = (approved_count / total_processed) * 100 if total_processed > 0 else 0
    
    print(f"\n**Results:** SESSION SUMMARY:")
    print(f"   EWA Requests Processed: {total_processed}")
    print(f"   Approval Rate: {approval_rate:.1f}%")
    print(f"   Regulatory Classification: [SUCCESS] All decisions tracked")
    print(f"   TILA Compliance: [SUCCESS] APR calculated when required")
    print(f"   CFPB Examination Ready: [SUCCESS] Complete audit trail")
    print(f"   Consumer Protection: [SUCCESS] Ability-to-repay assessed")

print(f"\n[ACCESS] CRITICAL REGULATORY ADVANTAGE:")
print(f"   EWA providers can operate with confidence in an uncertain")
print(f"   regulatory environment, knowing every decision is documented")
print(f"   with the correct classification and compliance requirements")
print(f"   that were applicable at the time of the decision.")

print(f"\n⚖ REGULATORY UNCERTAINTY PROTECTION:")
print(f"   As CFPB guidance continues to evolve, Briefcase AI provides")
print(f"   complete historical documentation showing compliance with")
print(f"   regulations as they were understood at each decision point.")

## Summary & Key Accomplishments

### [SUCCESS] What We Accomplished

1. **Regulatory Classification Tracking**: Implemented system to preserve EWA regulatory status at decision time as CFPB guidance evolves
2. **TILA Compliance Management**: Automated APR calculation and disclosure requirements based on current regulatory classification
3. **Ability-to-Repay Assessment**: Comprehensive evaluation of employee financial capacity for consumer protection
4. **Payroll Integration Verification**: Complete documentation of employer consent and wage calculation accuracy
5. **Temporal Regulatory Consistency**: Historical validation that decisions were compliant with guidance in effect at decision time

### **Objective:** Key Regulatory Benefits

- **CFPB Oversight Compliance**: Complete documentation for evolving regulatory interpretation of EWA products
- **TILA Disclosure Requirements**: Automated compliance with disclosure rules based on regulatory classification
- **Consumer Protection**: Built-in safeguards against excessive advance frequency and predatory practices
- **Examination Defense**: Historical documentation of regulatory compliance at decision time

### [ALERT] Critical Business Problem Solved

**The Problem**: EWA products exist in a regulatory gray area where classification continues to evolve with CFPB guidance, creating compliance uncertainty. Companies struggle to document that their historical decisions were compliant with regulations as they were understood at the time.

**The Solution**: Briefcase AI preserves the regulatory classification status and compliance requirements in effect at each decision point, creating a complete audit trail that demonstrates good faith compliance efforts regardless of how regulatory interpretation evolves.

### **Launch:** Production Implementation Guidance

1. **Payroll Integration**: Connect to employer payroll systems for real-time wage calculation
2. **Regulatory Monitoring**: Set up automated tracking of CFPB guidance changes and updates
3. **Compliance Automation**: Implement dynamic disclosure requirements based on current classification
4. **Consumer Protection**: Configure limits and safeguards based on ability-to-repay assessments
5. **Audit Trail Management**: Establish procedures for regulatory examination response using preserved decisions

### ⚖ Regulatory Evolution Preparedness

- **CFPB Guidance Changes**: Framework adapts to evolving EWA regulatory interpretation
- **TILA Requirement Evolution**: Disclosure requirements adjust automatically to classification changes
- **State-Level Regulation**: Structure supports state-specific EWA requirements as they develop
- **Consumer Protection Standards**: Framework accommodates increasing consumer protection focus

### **Reference:** Next Steps

- Integrate with your EWA platform and payroll partner APIs
- Customize regulatory classification monitoring for your specific interpretation and legal counsel guidance
- Set up automated CFPB guidance monitoring and compliance requirement updates
- Train compliance staff on temporal regulatory tracking and examination response procedures
- Establish ongoing monitoring for consumer protection metrics and advance frequency patterns

---

**[SECURED] Compliance Note**: This implementation demonstrates audit trail patterns for EWA regulatory compliance in an uncertain and evolving regulatory environment. EWA classification and TILA applicability continue to evolve through CFPB guidance, enforcement actions, and potential legislation. Always validate current regulatory interpretation with qualified consumer financial services law counsel specializing in earned wage access products.